In [ ]:
Explain this workflow aloud:
- User request: The user asks a finance question.
- Authorization: The application checks the user’s role and company-code access.
- Retrieval: It finds only documents the user is permitted to see.
- Prompt: It combines the question, retrieved information, and governance instructions.
- Model provider: SAP Generative AI Hub, Databricks, or the local provider generates a draft answer.
- Approval decision: The application validates the answer and determines whether human approval is required.
- Audit log: It records privacy-safe evidence such as the request ID, role, result, and policy decision.

# Find and prepare the project

In [1]:
# DAY 20 — PROJECT AND ENVIRONMENT SETUP
#
# Purpose:
# Locate the existing Git repository, confirm Python 3.11 and create
# the folders needed for today's implementation.
#
# Save this notebook somewhere inside the repository before running it.

from pathlib import Path
import importlib
import json
import os
import py_compile
import sys


def find_project_root(starting_path: Path) -> Path:
    """
    Search the current folder and its parents for the Git repository.

    This allows the notebook to run from either the repository root
    or a notebooks subfolder without hard-coding an operating-system path.
    """

    for candidate in [starting_path, *starting_path.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "The notebook is not inside the Git repository. "
        "Move it into the sap-finance-ai-assistant repository and rerun this cell."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Make all relative paths start from the repository root.
os.chdir(PROJECT_ROOT)

# Ensure modules in the app folder can be imported.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create today's output folders without deleting existing content.
for folder_name in ["app", "tests", "notes", "notebooks"]:
    (PROJECT_ROOT / folder_name).mkdir(parents=True, exist_ok=True)

# Confirm that the selected notebook kernel uses Python 3.11.
assert sys.version_info[:2] == (3, 11), (
    f"Expected Python 3.11, but found {sys.version.split()[0]}"
)

print("Project root:", PROJECT_ROOT)
print("Python version:", sys.version.split()[0])
print("✅ Day 20 environment is ready.")

Project root: /var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant
Python version: 3.11.17
✅ Day 20 environment is ready.


# Create the resilient API client

In [2]:
# DAY 20 — RESILIENT MODEL API CLIENT
#
# Purpose:
# Create a provider-neutral API boundary that can later call either
# Databricks Model Serving or SAP Generative AI Hub.
#
# The client provides:
# - request and idempotency identifiers;
# - retries for temporary failures;
# - immediate failure for permanent HTTP errors;
# - secret redaction;
# - privacy-aware audit metadata;
# - dependency injection for safe offline testing.

module_code = r'''
"""Resilient and auditable model API client."""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from hashlib import sha256
import time
from typing import Any, Callable
from uuid import uuid4


# These failures are commonly temporary and can be retried.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

# These keys must never appear in an audit event with their real values.
SENSITIVE_KEYS = {
    "authorization",
    "api_key",
    "apikey",
    "password",
    "secret",
    "token",
}


@dataclass(frozen=True)
class TransportResponse:
    """Provider-neutral representation of an HTTP response."""

    status_code: int
    body: dict[str, Any] = field(default_factory=dict)
    headers: dict[str, str] = field(default_factory=dict)


@dataclass(frozen=True)
class ApiCallResult:
    """Normalized result returned to the finance application."""

    ok: bool
    request_id: str
    attempts: int
    status_code: int | None
    response: dict[str, Any] | None
    error_code: str | None
    audit_event: dict[str, Any]


def redact_sensitive_values(value: Any) -> Any:
    """
    Recursively replace credential-like values before audit logging.

    The application can still use the original metadata for its request,
    but the audit event receives only the redacted copy.
    """

    if isinstance(value, dict):
        redacted = {}

        for key, item in value.items():
            normalized_key = str(key).lower().replace("-", "_")

            sensitive_suffix = normalized_key.endswith(
                ("_token", "_secret", "_password", "_api_key")
            )

            if normalized_key in SENSITIVE_KEYS or sensitive_suffix:
                redacted[key] = "[REDACTED]"
            else:
                redacted[key] = redact_sensitive_values(item)

        return redacted

    if isinstance(value, (list, tuple)):
        return [redact_sensitive_values(item) for item in value]

    return value


class ScriptedTransport:
    """
    Deterministic fake API transport used for notebook testing.

    Each scripted step is either a TransportResponse or an exception.
    No real provider, internet connection or API credential is required.
    """

    def __init__(self, steps: list[Any]):
        self.steps = list(steps)
        self.calls: list[dict[str, Any]] = []

    def __call__(
        self,
        payload: dict[str, Any],
        headers: dict[str, str],
    ) -> TransportResponse:
        self.calls.append(
            {
                "payload": payload,
                "headers": headers.copy(),
            }
        )

        if not self.steps:
            raise RuntimeError("The scripted transport has no remaining response.")

        next_step = self.steps.pop(0)

        if isinstance(next_step, BaseException):
            raise next_step

        return next_step


class ResilientModelClient:
    """
    Call a model provider through an injected transport function.

    A real implementation can replace ScriptedTransport with an adapter
    for Databricks Model Serving or SAP Generative AI Hub.
    """

    def __init__(
        self,
        transport: Callable[
            [dict[str, Any], dict[str, str]],
            TransportResponse,
        ],
        max_attempts: int = 3,
        base_delay_seconds: float = 0.25,
        sleep_function: Callable[[float], None] = time.sleep,
    ):
        if max_attempts < 1:
            raise ValueError("max_attempts must be at least 1.")

        self.transport = transport
        self.max_attempts = max_attempts
        self.base_delay_seconds = base_delay_seconds
        self.sleep_function = sleep_function

    def _retry_delay(
        self,
        attempt: int,
        response: TransportResponse | None = None,
    ) -> float:
        """
        Prefer the provider's Retry-After value when supplied.

        Otherwise use exponential backoff:
        base delay, 2 × base delay, 4 × base delay, and so forth.
        """

        if response is not None:
            retry_after = response.headers.get("Retry-After")

            if retry_after is not None:
                try:
                    return max(0.0, float(retry_after))
                except ValueError:
                    pass

        return self.base_delay_seconds * (2 ** (attempt - 1))

    def _build_result(
        self,
        *,
        ok: bool,
        request_id: str,
        attempts: int,
        status_code: int | None,
        response_body: dict[str, Any] | None,
        error_code: str | None,
        user_role: str,
        question: str,
        approved_context: list[dict[str, Any]],
        metadata: dict[str, Any],
    ) -> ApiCallResult:
        """
        Build a result and its privacy-aware audit event.

        The raw question and retrieved policy text are deliberately excluded.
        A SHA-256 hash supports correlation without storing the question.
        """

        policy_ids = sorted(
            {
                str(item["policy_id"])
                for item in approved_context
                if item.get("policy_id")
            }
        )

        audit_event = {
            "event_type": "model_api_call",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "request_id": request_id,
            "user_role": user_role,
            "question_sha256": sha256(
                question.encode("utf-8")
            ).hexdigest(),
            "policy_ids": policy_ids,
            "attempts": attempts,
            "status_code": status_code,
            "outcome": "success" if ok else "failure",
            "error_code": error_code,
            "metadata": redact_sensitive_values(metadata),
        }

        return ApiCallResult(
            ok=ok,
            request_id=request_id,
            attempts=attempts,
            status_code=status_code,
            response=response_body if ok else None,
            error_code=error_code,
            audit_event=audit_event,
        )

    def invoke(
        self,
        *,
        question: str,
        user_role: str,
        approved_context: list[dict[str, Any]],
        metadata: dict[str, Any] | None = None,
        request_id: str | None = None,
    ) -> ApiCallResult:
        """Invoke the provider with bounded retry behavior."""

        metadata = metadata or {}
        request_id = request_id or str(uuid4())

        # The same identifiers are reused for every retry of one logical call.
        headers = {
            "X-Request-ID": request_id,
            "Idempotency-Key": request_id,
        }

        payload = {
            "question": question,
            "user_role": user_role,
            "approved_context": approved_context,
            "metadata": metadata,
        }

        for attempt in range(1, self.max_attempts + 1):
            try:
                provider_response = self.transport(payload, headers)

            except (TimeoutError, ConnectionError) as error:
                if attempt < self.max_attempts:
                    delay = self._retry_delay(attempt)
                    self.sleep_function(delay)
                    continue

                return self._build_result(
                    ok=False,
                    request_id=request_id,
                    attempts=attempt,
                    status_code=None,
                    response_body=None,
                    error_code=type(error).__name__.lower(),
                    user_role=user_role,
                    question=question,
                    approved_context=approved_context,
                    metadata=metadata,
                )

            status_code = provider_response.status_code

            if 200 <= status_code < 300:
                return self._build_result(
                    ok=True,
                    request_id=request_id,
                    attempts=attempt,
                    status_code=status_code,
                    response_body=provider_response.body,
                    error_code=None,
                    user_role=user_role,
                    question=question,
                    approved_context=approved_context,
                    metadata=metadata,
                )

            retryable = status_code in RETRYABLE_STATUS_CODES

            if retryable and attempt < self.max_attempts:
                delay = self._retry_delay(attempt, provider_response)
                self.sleep_function(delay)
                continue

            return self._build_result(
                ok=False,
                request_id=request_id,
                attempts=attempt,
                status_code=status_code,
                response_body=None,
                error_code=f"http_{status_code}",
                user_role=user_role,
                question=question,
                approved_context=approved_context,
                metadata=metadata,
            )

        raise AssertionError("The retry loop ended unexpectedly.")
'''

module_path = PROJECT_ROOT / "app" / "resilient_model_client.py"
module_path.write_text(module_code.lstrip(), encoding="utf-8")

# Compile the new module immediately to identify syntax errors.
py_compile.compile(str(module_path), doraise=True)

print("Created:", module_path.relative_to(PROJECT_ROOT))
print("✅ Resilient API client compiled successfully.")

Created: app/resilient_model_client.py
✅ Resilient API client compiled successfully.


# Run six deterministic API scenarios

In [3]:
# DAY 20 — API RESILIENCE TESTS
#
# Purpose:
# Verify successful calls, retries, fail-fast behavior, request-ID reuse
# and audit-log redaction without calling a live model provider.

from dataclasses import asdict

# Reload the module so rerunning the notebook uses the latest saved code.
importlib.invalidate_caches()
sys.modules.pop("app.resilient_model_client", None)

from app.resilient_model_client import (
    ResilientModelClient,
    ScriptedTransport,
    TransportResponse,
)


def successful_response() -> TransportResponse:
    """Return a grounded synthetic finance answer."""

    return TransportResponse(
        status_code=200,
        body={
            "answer": (
                "Manual journal entries above the policy threshold "
                "require finance-manager approval."
            ),
            "decision": "answer",
            "requires_human_approval": True,
            "citations": ["FIN-JE-001"],
        },
    )


# Each scenario has an expected outcome, number of attempts and final status.
scenario_specs = [
    {
        "name": "success_first_attempt",
        "steps": [successful_response()],
        "expected_ok": True,
        "expected_attempts": 1,
        "expected_status": 200,
    },
    {
        "name": "rate_limit_then_success",
        "steps": [
            TransportResponse(
                429,
                {"error": "rate_limited"},
                {"Retry-After": "0"},
            ),
            successful_response(),
        ],
        "expected_ok": True,
        "expected_attempts": 2,
        "expected_status": 200,
    },
    {
        "name": "service_unavailable_then_success",
        "steps": [
            TransportResponse(503, {"error": "unavailable"}),
            successful_response(),
        ],
        "expected_ok": True,
        "expected_attempts": 2,
        "expected_status": 200,
    },
    {
        "name": "unauthorized_no_retry",
        "steps": [
            TransportResponse(401, {"error": "unauthorized"}),
        ],
        "expected_ok": False,
        "expected_attempts": 1,
        "expected_status": 401,
    },
    {
        "name": "timeout_then_success",
        "steps": [
            TimeoutError("Synthetic provider timeout"),
            successful_response(),
        ],
        "expected_ok": True,
        "expected_attempts": 2,
        "expected_status": 200,
    },
    {
        "name": "retry_attempts_exhausted",
        "steps": [
            TransportResponse(500, {"error": "internal_error"}),
            TransportResponse(502, {"error": "bad_gateway"}),
            TransportResponse(503, {"error": "unavailable"}),
        ],
        "expected_ok": False,
        "expected_attempts": 3,
        "expected_status": 503,
    },
]

approved_context = [
    {
        "policy_id": "FIN-JE-001",
        "text": (
            "Manual journal entries above USD 100,000 require "
            "finance-manager approval."
        ),
    }
]

question = "What approval is required for a USD 125,000 journal entry?"

day20_results = []

for specification in scenario_specs:
    transport = ScriptedTransport(specification["steps"])

    # Disable real waiting so retry tests complete immediately.
    recorded_delays = []

    client = ResilientModelClient(
        transport=transport,
        max_attempts=3,
        base_delay_seconds=0,
        sleep_function=recorded_delays.append,
    )

    result = client.invoke(
        question=question,
        user_role="finance_analyst",
        approved_context=approved_context,
        metadata={
            "provider": "provider-neutral-test",
            "prompt_template_id": "governed-sap-finance-assistant",
            "prompt_template_version": "1.0.0",
            "api_token": "not-for-logging",
        },
    )

    # Every retry must keep the same correlation and idempotency identifiers.
    request_ids = {
        call["headers"]["X-Request-ID"]
        for call in transport.calls
    }

    headers_valid = (
        request_ids == {result.request_id}
        and all(
            call["headers"]["Idempotency-Key"] == result.request_id
            for call in transport.calls
        )
    )

    # Confirm that the secret is absent from the audit record.
    audit_text = json.dumps(result.audit_event)

    audit_is_safe = (
        "not-for-logging" not in audit_text
        and result.audit_event["metadata"]["api_token"] == "[REDACTED]"
        and question not in audit_text
    )

    passed = all(
        [
            result.ok is specification["expected_ok"],
            result.attempts == specification["expected_attempts"],
            result.status_code == specification["expected_status"],
            headers_valid,
            audit_is_safe,
        ]
    )

    day20_results.append(
        {
            "scenario": specification["name"],
            "passed": passed,
            "expected_ok": specification["expected_ok"],
            "expected_attempts": specification["expected_attempts"],
            "expected_status": specification["expected_status"],
            "recorded_delays": recorded_delays,
            "result": asdict(result),
        }
    )

# Print a compact result for every test.
for test_result in day20_results:
    status = "PASS" if test_result["passed"] else "FAIL"
    actual = test_result["result"]

    print(
        f"{status:4} | "
        f"{test_result['scenario']:<34} | "
        f"attempts={actual['attempts']} | "
        f"status={actual['status_code']}"
    )

PASS | success_first_attempt              | attempts=1 | status=200
PASS | rate_limit_then_success            | attempts=2 | status=200
PASS | service_unavailable_then_success   | attempts=2 | status=200
PASS | unauthorized_no_retry              | attempts=1 | status=401
PASS | timeout_then_success               | attempts=2 | status=200
PASS | retry_attempts_exhausted           | attempts=3 | status=503


# Inspect success and failure envelopes

In [4]:
# DAY 20 — RESULT INSPECTION
#
# Purpose:
# Compare a recovered API call with a permanent authentication failure.
#
# Notice that both outcomes have the same predictable application structure.

recovered_call = next(
    item
    for item in day20_results
    if item["scenario"] == "rate_limit_then_success"
)

rejected_call = next(
    item
    for item in day20_results
    if item["scenario"] == "unauthorized_no_retry"
)

print("RECOVERED CALL")
print(json.dumps(recovered_call["result"], indent=2))

print("\nPERMANENT FAILURE")
print(json.dumps(rejected_call["result"], indent=2))

RECOVERED CALL
{
  "ok": true,
  "request_id": "5a70b799-e8fd-4cff-8d98-62851e7c947c",
  "attempts": 2,
  "status_code": 200,
  "response": {
    "answer": "Manual journal entries above the policy threshold require finance-manager approval.",
    "decision": "answer",
    "requires_human_approval": true,
    "citations": [
      "FIN-JE-001"
    ]
  },
  "error_code": null,
  "audit_event": {
    "event_type": "model_api_call",
    "timestamp_utc": "2026-10-28T03:44:49.772865+00:00",
    "request_id": "5a70b799-e8fd-4cff-8d98-62851e7c947c",
    "user_role": "finance_analyst",
    "question_sha256": "967f539f036b218639da765d1ca10bae8a07b26ba6e1b183b25bbf684f83cb64",
    "policy_ids": [
      "FIN-JE-001"
    ],
    "attempts": 2,
    "status_code": 200,
    "outcome": "success",
    "error_code": null,
    "metadata": {
      "provider": "provider-neutral-test",
      "prompt_template_id": "governed-sap-finance-assistant",
      "prompt_template_version": "1.0.0",
      "api_token": "[R

# Save test evidence and engineering notes

In [5]:
# DAY 20 — SAVE EVIDENCE AND LEARNING NOTES
#
# Purpose:
# Preserve machine-readable test evidence and a human-readable explanation
# suitable for GitHub, interview preparation and portfolio review.

passed_count = sum(
    test_result["passed"]
    for test_result in day20_results
)

report = {
    "day": 20,
    "date": "2026-10-26",
    "topic": "Python APIs and resilient model invocation",
    "module": "app/resilient_model_client.py",
    "total_cases": len(day20_results),
    "passed_cases": passed_count,
    "cases": day20_results,
}

report_path = (
    PROJECT_ROOT
    / "tests"
    / "day20_api_resilience_results.json"
)

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

notes_text = f"""# Day 20 — Resilient Model API Calls

## Objective

Add a reliable, provider-neutral API boundary to the governed SAP Finance
assistant.

## What I implemented

- Bounded retries for HTTP 429, 500, 502, 503 and 504.
- Retry handling for timeouts and connection errors.
- Fail-fast behavior for non-retryable errors such as HTTP 401.
- Correlation and idempotency identifiers.
- Normalized success and failure envelopes.
- Question hashing and credential redaction in audit events.
- Six deterministic tests requiring no live API credentials.

## Results

- Tests passed: {passed_count}/{len(day20_results)}
- Generated module: `app/resilient_model_client.py`
- Generated evidence: `tests/day20_api_resilience_results.json`

## Architecture decision

The finance application depends on a provider-neutral client instead of
calling a model endpoint directly. A later adapter can connect the same
application to Databricks Model Serving or SAP Generative AI Hub while
preserving consistent retry, error and audit behavior.

## Interview explanation

I built the API boundary around the model, not merely a model call. The
client distinguishes temporary failures from permanent failures, preserves
one request ID across retries, prevents unlimited retries, and records
privacy-aware audit evidence. This improves reliability without weakening
financial governance.
"""

notes_path = (
    PROJECT_ROOT
    / "notes"
    / "day20_python_api_resilience.md"
)

notes_path.write_text(notes_text, encoding="utf-8")

print("Saved:", report_path.relative_to(PROJECT_ROOT))
print("Saved:", notes_path.relative_to(PROJECT_ROOT))
print(f"Tests passed: {passed_count}/{len(day20_results)}")

Saved: tests/day20_api_resilience_results.json
Saved: notes/day20_python_api_resilience.md
Tests passed: 6/6


# Single completion check

In [7]:
# DAY 20 — COMPLETION CHECK
#
# Purpose:
# Declare today's assignment complete only if the module compiles,
# every scenario passes and both evidence files exist.

py_compile.compile(str(module_path), doraise=True)

required_outputs = [
    module_path,
    report_path,
    notes_path,
]

assert all(path.exists() for path in required_outputs), (
    "One or more Day 20 output files are missing."
)

assert len(day20_results) == 6, (
    "Expected exactly six API resilience scenarios."
)

assert all(item["passed"] for item in day20_results), (
    "At least one API resilience scenario failed."
)

print("✅ DAY 20 COMPLETE — 6/6 API resilience scenarios passed.")


✅ DAY 20 COMPLETE — 6/6 API resilience scenarios passed.


In [8]:
from pathlib import Path
from IPython.display import HTML, display
import base64


file_name = "day20_python_api_resilience.ipynb"

matches = [
    path
    for path in Path.cwd().rglob(file_name)
    if ".ipynb_checkpoints" not in path.parts
]

if not matches:
    raise FileNotFoundError(
        f"Could not locate {file_name}"
    )

notebook_path = matches[0]
encoded_file = base64.b64encode(
    notebook_path.read_bytes()
).decode("utf-8")

display(
    HTML(
        f"""
        <a
            href="data:application/x-ipynb+json;base64,{encoded_file}"
            download="{file_name}"
            style="
                display:inline-block;
                padding:12px 18px;
                background:#1677ff;
                color:white;
                text-decoration:none;
                border-radius:6px;
                font-weight:bold;
            "
        >
            Download Day 20 Notebook
        </a>
        """
    )
)

print("Notebook located at:", notebook_path)

Notebook located at: /var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/day20_python_api_resilience.ipynb
